# 🏥 삼성화재 보험 AI 에이전트 - 라우터

```
사용자 로그인
    ↓
🧭 라우터 에이전트     → 질문 의도 분류 + 도메인 선별
    ↓
👤 customer_agent     → 고객 DB 조회
📚 rag_agent          → 약관 검색 + 답변 생성
🚗 claim_agent        → 서류 접수 + 심사
🚨 complaint_agent    → 불만 감지 + 민원 DB 저장
```

In [1]:
import sys, json
from pathlib import Path

sys.path.insert(0, str(Path().resolve()))

from utils.llm_setup import llm, PRODUCT_TO_DOMAIN
from agents.customer_agent import login, format_customer_info, get_subscribed_domains
from agents.rag_agent import search_and_answer
from agents.claim_agent import handle_claim
from agents.complaint_agent import check_and_record
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

print('✅ 모든 에이전트 임포트 완료')

⏳ BGE-m3 임베딩 모델 로드 중...
⏳ 4개의 전문 DB 로드 중...
  ✅ 자동차보험: 1344개 청크
  ✅ 암보험: 1780개 청크
  ✅ 치아보험: 1158개 청크
  ✅ 판례: 96개 청크
✅ 모든 세팅 완료!

✅ 모든 에이전트 임포트 완료


In [2]:
# ==========================================
# 1. 로그인
# ==========================================
CUSTOMER_ID = "CUST-0004"   # ← 테스트할 고객 ID
PASSWORD    = "1234"

customer_info = login(CUSTOMER_ID, PASSWORD)

if customer_info:
    print(f"✅ 로그인 성공: {customer_info['name']}님")
    print()
    print("[가입 보험 목록]")
    for p in customer_info["policies"]:
        print(f"  - {p['product_name']} ({p['joined_year']}년 가입) | 한도: {p['coverage_limit']} | 특약: {p['riders']}")
else:
    print("❌ 로그인 실패: ID 또는 비밀번호를 확인하세요.")

✅ 로그인 성공: 최하은님

[가입 보험 목록]
  - 치아보험 플랜D (2019년 가입) | 한도: 500만원 | 특약: 크라운특약;보철치료특약


In [3]:
# ==========================================
# 2. 라우터 에이전트 - 질문 의도 분류
# ==========================================

# 욕설/부적절 키워드 필터
BLOCKED_KEYWORDS = ["씨발", "개새끼", "병신", "존나", "ㅅㅂ", "ㅂㅅ"]

def is_abusive(query: str) -> bool:
    return any(k in query for k in BLOCKED_KEYWORDS)

INTENT_ROUTER_PROMPT = """
당신은 삼성화재 보험 고객센터 AI의 질문 분류기입니다.

[로그인 고객 정보]
{customer_info}

[고객 질문]
{query}

아래 JSON 형식으로만 응답하세요. 다른 말은 절대 금지입니다.

{{
  "intent": "보장조회 또는 미가입문의 또는 사고청구 또는 일반문의 또는 out_of_scope",
  "target_domain": ["auto", "cancer", "teeth", "precedent" 중 해당하는 것만],
  "is_subscribed": true 또는 false,
  "needs_document_guide": true 또는 false,
  "sub_queries": ["약관 검색에 쓸 핵심 질문들"]
}}

분류 기준:
- 보장조회: 가입한 보험의 보장 범위, 지급 여부 질문 ("되나요", "받을 수 있나요", "보장되나요")
- 미가입문의: 아직 가입하지 않은 보험에 대한 설명/가입 문의
- 사고청구: 실제 사고 발생 또는 보험금 청구 의사 표현 ("사고났어", "청구하려고", "신청하려고")
- 일반문의: 보험과 관련된 일반적인 질문
- out_of_scope: 보험과 전혀 관련 없는 질문 (날씨, 음식, 연예인, 주식, 정치 등)
- needs_document_guide: 사고청구인 경우 true
- 보장조회: 가입한 보험의 보장 범위, 지급 여부 질문
  + "내가 ~보험 가입했는데 ~되나요" 형태도 보장조회로 분류
  + "내 보험으로 ~가능한가요"
  + "가입했는데 ~보장되나요"
  + 고객이 자신의 가입 여부를 언급하며 보장을 묻는 경우 모두 보장조회
"""

def router_agent(customer_info: dict, user_query: str) -> dict:
    """질문을 분석하여 라우팅 결과 반환"""
    customer_info_str = format_customer_info(customer_info)

    prompt = PromptTemplate.from_template(INTENT_ROUTER_PROMPT)
    chain = prompt | llm | StrOutputParser()
    result_str = chain.invoke({
        "customer_info": customer_info_str,
        "query": user_query
    })

    result_str = result_str.strip().replace("```json", "").replace("```", "")
    routing = json.loads(result_str)

    domain_kr = {"auto": "자동차보험", "cancer": "암보험", "teeth": "치아보험", "precedent": "판례"}
    print(f"  🧭 의도       : {routing['intent']}")
    print(f"  📋 도메인     : {[domain_kr.get(d, d) for d in routing['target_domain']]}")
    print(f"  ✅ 가입여부   : {'가입' if routing['is_subscribed'] else '미가입'}")
    print(f"  📄 서류안내   : {'필요' if routing['needs_document_guide'] else '불필요'}")

    return routing

print('✅ 라우터 에이전트 세팅 완료')

✅ 라우터 에이전트 세팅 완료


In [11]:
# ==========================================
# 3. 라우팅 실행 함수
# ==========================================
def execute(user_query: str, image_paths: list = None) -> str:
    """
    라우팅 결과에 따라 적절한 에이전트로 분기

    Args:
        user_query  : 고객 질문 텍스트
        image_paths : 서류 이미지 경로 리스트 (청구 시 사용)
    """
    if not customer_info:
        return "❌ 로그인이 필요합니다."

    print(f"\n🧐 [{customer_info['name']}] 질문: {user_query}")
    print("=" * 60)

    # ── Step 0: 욕설/부적절 입력 필터 ──────────────
    if is_abusive(user_query):
        return "⚠️ 부적절한 표현이 포함되어 있어 답변드리기 어렵습니다.\n정중한 표현으로 다시 질문해 주시면 성심껏 도와드리겠습니다."

    # ── Step 1: 라우터로 의도 분류 ─────────────────
    routing = router_agent(customer_info, user_query)
    intent  = routing["intent"]
    domains = routing["target_domain"]
    customer_context = format_customer_info(customer_info)

    print()

    # ── Step 2: 의도별 분기 ────────────────────────

    # 보험 무관 질문
    # out_of_scope 케이스 수정
    if intent == "out_of_scope":
        # LLM이 자연스럽게 답변 + 보험 유도 멘트
        out_of_scope_prompt = PromptTemplate.from_template("""
    고객이 보험과 관련 없는 질문을 했습니다.
    친근하고 자연스럽게 짧게 답변한 뒤, 마지막에 보험 관련 질문을 유도하는 멘트로 마무리하세요.
    보험 유도 멘트 예시: "혹시 보험 관련해서 궁금하신 점이 있으시면 편하게 말씀해 주세요 😊"

    고객 질문: {query}
    답변:""")
        chain = out_of_scope_prompt | llm | StrOutputParser()
        answer = chain.invoke({"query": user_query})

    # 보장조회
    elif intent == "보장조회":
        answer = search_and_answer(
            query=user_query,
            domains=domains,
            customer_context=customer_context
        )

    # 미가입 보험 문의
    elif intent == "미가입문의":
        answer = search_and_answer(
            query=user_query,
            domains=domains,
            customer_context=""  # 미가입이므로 고객 정보 제외
        )
        answer += "\n\n📌 가입을 원하시면 삼성화재 홈페이지(www.samsungfire.com)에서 가입하실 수 있습니다."

    # 사고/청구
    elif intent == "사고청구":
        domain = domains[0] if domains else "auto"

        # 보장 내용 확인
        coverage_answer = search_and_answer(
            query=user_query,
            domains=domains,
            customer_context=customer_context
        )

        # 서류 접수 + 검증
        claim_answer = handle_claim(
            customer_info=customer_info,
            domain=domain,
            query=user_query,
            image_paths=image_paths  # 이미지 있으면 분류 후 검증, 없으면 서류 안내
        )

        answer = f"{coverage_answer}\n\n{'─'*40}\n\n{claim_answer}"

    # 일반문의 (보험 관련)
    else:
        answer = search_and_answer(
            query=user_query,
            domains=domains,
            customer_context=""
        )

    # ── Step 3: 불만 감지 (out_of_scope, 욕설 제외) ─
    if intent not in ("out_of_scope",):
        complaint_msg = check_and_record(customer_info, user_query, answer)
        if complaint_msg:
            answer += complaint_msg

    return answer

print('✅ 실행 함수 세팅 완료')

✅ 실행 함수 세팅 완료


---
## 🧪 테스트 시나리오

In [5]:
# 시나리오 1: 보장조회
result = execute("가입 3달 전에 아프던 이가 지금 보상되나요?")
print(result)


🧐 [최하은] 질문: 가입 3달 전에 아프던 이가 지금 보상되나요?
  🧭 의도       : 보장조회
  📋 도메인     : ['치아보험']
  ✅ 가입여부   : 가입
  📄 서류안내   : 불필요

  ⚖️  관련 판례 없음 → 판례 컨텍스트 미포함
최하은 고객님, 삼성화재 최고의 보험 보상 심사역이자 법률 자문 에이전트입니다. 고객님의 치아보험 플랜D(2019년 가입, 7년차)에 대한 문의에 명확하게 답변드리겠습니다.

고객님께서 가입 3달 전에 이미 아프던 이에 대한 보상 가능 여부를 문의주셨습니다. 이는 보험 가입 전 발생했거나 인지하고 있던 질병(치아 상태)에 대한 보상 여부와 관련된 중요한 내용입니다.

저희 삼성화재 치아보험 약관 및 분쟁 사례 안내에 따르면, **보험 가입 전 이미 발생했거나 치료가 필요했던 질병 또는 상해에 대해서는 보험금을 지급하지 않는 것을 원칙으로 합니다.**

구체적으로 다음 약관 내용을 근거로 설명드릴 수 있습니다.

1.  **[유형3] 보험계약 전 알릴 의무 위반** 사례에서 명시된 바와 같이, "보험 가입전 치료를 받고 있던 질병(재발 질병 포함)은 보험금을 지급받지 못한다"고 안내하고 있습니다. 고객님께서 가입 3달 전에 이미 아프던 이가 있었다면, 이는 보험 가입 전 이미 존재했던 질병 상태로 간주될 수 있습니다.
2.  **[유형6] 이미 크라운치료를 받은 치아에 대한 보장기준** 사례에서도, "보험 가입 전 이미 크라운치료를 받았던 부분이 파손되어 수리, 복구, 대체의 목적으로 새로운 크라운으로 재 장착 후 보험금을 청구하였으나 보험금이 거절됨"이라는 내용이 있습니다. 이는 보험 가입 전 발생한 치아 문제에 대한 치료는 보상되지 않는다는 원칙을 다시 한번 보여줍니다.

따라서, 고객님의 경우 가입 3달 전에 이미 아프던 이였다면, 이는 보험 가입 전 발생한 질병으로 보아 현재 치료를 받으시더라도 해당 치아에 대한 보험금 지급은 어렵습니다. 고객님께서 가입하신 크라운특약 및 보철치료특약은 보험 가입 후

In [6]:
# 시나리오 2: 미가입 보험 문의
result = execute("자동차보험 약관 좀 설명해줄 수 있어?")
print(result)


🧐 [최하은] 질문: 자동차보험 약관 좀 설명해줄 수 있어?
  🧭 의도       : 일반문의
  📋 도메인     : ['자동차보험']
  ✅ 가입여부   : 미가입
  📄 서류안내   : 불필요

  ⚖️  관련 판례 없음 → 판례 컨텍스트 미포함
  🚨 민원 감지 | 유형: 약관이해불만 | 점수: 4/10 | ID: CMP-20260510011441
고객님, 안녕하세요. 삼성화재 최고의 보험 보상 심사역이자 법률 자문 에이전트입니다.

고객님께서 문의하신 자동차보험 약관에 대해 명확하게 설명해 드리겠습니다.

**자동차보험 약관이란?**

자동차보험 약관은 고객님께서 자동차를 소유, 사용, 관리하는 동안 발생할 수 있는 사고에 대해 보상하는 보험의 구체적인 내용과, 보험계약의 성립부터 소멸까지 보험계약자와 보험회사 간의 권리 및 의무사항을 명시해 놓은 문서입니다. 특히, 고객님께서 가입하신 보험은 '개인용 자동차보험 약관'을 기준으로 합니다.

**약관을 쉽게 이해하는 꿀팁**

저희 삼성화재는 고객님께서 약관을 보다 쉽고 편리하게 이용하실 수 있도록 다음과 같은 꿀팁을 제공하고 있습니다.

*   **약관요약서 활용:** 시각화된 '약관요약서'(p.7)를 통해 계약 일반사항, 가입 시 유의사항, 민원사례 등을 쉽게 이해하실 수 있습니다.
*   **용어의 정의 및 설명 참고:** 약관 내용 중 어려운 보험 용어는 '용어의 정의', '약관 본문 내용 설명 및 예시'(p.51)를 참고하시면 이해에 도움이 됩니다.
*   **QR코드 활용:** 스마트폰으로 QR코드를 인식하면 약관 해설 동영상, 보험금 지급 절차, 전국 지점 등을 쉽게 안내받을 수 있습니다.(p.4)
*   **주요 내용 확인:** 약관 조항 등이 음영·컬러화 되거나 진하게 표시된 부분은 보험금 지급 등 약관의 주요 내용이므로 주의 깊게 읽어보시는 것이 중요합니다.

**자동차보험 약관의 핵심 체크 항목 (보통약관 기준)**

약관에는 크게 '보상하는 손해', '보상하지 않는 손해', 그

In [7]:
# 시나리오 3: 사고청구 (서류 없이 → 서류 안내)
result = execute("어제 접촉사고가 났는데 얼마까지 보상받을 수 있지?")
print(result)


🧐 [최하은] 질문: 어제 접촉사고가 났는데 얼마까지 보상받을 수 있지?
  🧭 의도       : 사고청구
  📋 도메인     : ['자동차보험']
  ✅ 가입여부   : 미가입
  📄 서류안내   : 필요

  ⚖️  관련 판례 없음 → 판례 컨텍스트 미포함
  🚨 민원 감지 | 유형: 약관이해불만 | 점수: 3/10 | ID: CMP-20260510011515
최하은 고객님, 삼성화재 최고의 보험 보상 심사역이자 법률 자문 에이전트입니다. 어제 발생한 접촉사고로 인해 보상 가능 여부와 한도에 대해 문의주셨습니다.

고객님의 가입 정보를 확인한 결과, 현재 삼성화재에는 '치아보험 플랜D' (가입연도 2019년, 7년차)만 가입되어 있으십니다. 치아보험은 치아 치료에 대한 보장을 목적으로 하는 상품으로, 자동차 접촉사고로 인한 손해나 상해에 대한 보상은 포함되지 않습니다.

제공된 검색 자료는 '자동차보험 약관'에 대한 내용으로, 교통사고 발생 시 형사합의금, 심리상담 지원금, 상해 및 장애 등급별 보상 한도에 대해 설명하고 있습니다. 그러나 이 약관은 고객님께서 가입하신 치아보험과는 별개의 상품입니다.

따라서 고객님의 현재 가입 정보(치아보험 플랜D)만으로는 접촉사고에 대한 보상을 삼성화재에서 직접 처리해 드리기 어렵습니다. 만약 별도로 자동차보험에 가입되어 있으시다면 해당 보험의 약관에 따라 보상 여부 및 한도를 확인해 보셔야 합니다. 사고 상대방이 있으시다면 상대방의 자동차보험을 통해 보상을 받으실 수도 있습니다.

────────────────────────────────────────

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📋 자동차보험 보험금 청구 안내
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
최하은님의 가입 특약()을 포함한 필요 서류입니다:

  1. 사고접수확인서
  2. 자동차사고사실확인원
  3. 진단서 또는 소견서
  4. 수리 견적서 또는 수리 영수증
  5. 차

In [8]:
# 시나리오 4: 사고청구 (이미지 제출)
result = execute(
    "서류 제출할게요",
    image_paths=["test_images/진단서.jpg", "test_images/영수증.png"]
)
print(result)


🧐 [최하은] 질문: 서류 제출할게요
  🧭 의도       : 사고청구
  📋 도메인     : ['치아보험']
  ✅ 가입여부   : 가입
  📄 서류안내   : 필요

  ⚖️  관련 판례 없음 → 판례 컨텍스트 미포함
  🖼️  이미지 2개 분석 중...


C:\Users\cloud\Desktop\2026_new\보험 AI 공모전\통합\agents\claim_agent.py:176: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


FileNotFoundError: [Errno 2] No such file or directory: 'test_images\\진단서.jpg'

In [12]:
# 시나리오 5: 보험 무관 질문
result = execute("오늘 날씨 어때?")
print(result)


🧐 [최하은] 질문: 오늘 날씨 어때?
  🧭 의도       : out_of_scope
  📋 도메인     : []
  ✅ 가입여부   : 미가입
  📄 서류안내   : 불필요

오늘 날씨 정말 좋네요! 햇살도 따뜻하고 바람도 선선해서 기분까지 좋아지는 날씨예요 😊 혹시 보험 관련해서 궁금하신 점이 있으시면 편하게 말씀해 주세요 😊


In [10]:
# 시나리오 6: 부적절한 입력
result = execute("씨발 왜 보상이 안 되는 거야")
print(result)


🧐 [최하은] 질문: 씨발 왜 보상이 안 되는 거야
⚠️ 부적절한 표현이 포함되어 있어 답변드리기 어렵습니다.
정중한 표현으로 다시 질문해 주시면 성심껏 도와드리겠습니다.


In [ ]:
# 대화형 루프
# 이미지 제출: "파일: 경로1, 경로2" 형식으로 입력
print(f"💬 {customer_info['name']}님, 무엇이 궁금하신가요? (종료: q)")
print("💡 서류 제출 시: 파일: C:/경로/진단서.jpg, C:/경로/영수증.png\n")

while True:
    query = input("질문: ").strip()
    if query.lower() == "q":
        break
    if not query:
        continue

    # "파일:"로 시작하면 이미지 경로로 처리
    if query.startswith("파일:"):
        paths = [p.strip() for p in query[3:].split(",") if p.strip()]
        print()
        print(execute("서류 제출할게요", image_paths=paths))
    else:
        print()
        print(execute(query))
    print()